# Lesson: Einops Reshaping Drills

Goal: Get fluent with einops for tensor reshaping/manipulation - a skill ARENA explicitly calls out as prerequisite material, and one that shows up constantly once you get into transformers (patch embeddings, multi-head attention reshaping, etc).

These tasks were generated by Claude for me to learn on

In [ ]:
import torch
from einops import rearrange, reduce, repeat

/home/fin/doccuments/work/mech-interp-mini-projects/einops_drills/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## Part 1: Warm-Up Drills



Drill 1. x has shape [batch, channels, height, width]. Convert to [batch, height, width, channels] (NCHW → NHWC).

In [ ]:
x1 = torch.randn(2, 3, 32, 32)  # [batch, channels, height, width]
print(x1.shape)
x1 = rearrange(x1, "batch channels height width -> batch height width channels")
x1.shape

torch.Size([2, 3, 32, 32])


torch.Size([2, 32, 32, 3])

Drill 2. x has shape [batch, height, width, channels]. Flatten the spatial dimensions into one axis: [batch, height*width, channels].

In [ ]:
x2 = torch.randn(2, 3, 32, 32)  # [batch, channels, height, width]
print(x2.shape)
x2 = rearrange(x2, "batch channels height width -> batch (height width) channels")
x2.shape

torch.Size([2, 32, 32, 3])


torch.Size([2, 96, 32])

Drill 3. x has shape [batch, seq_len, embed_dim], and embed_dim = num_heads * head_dim. Split it into separate heads: [batch, num_heads, seq_len, head_dim].

In [ ]:
x3 = torch.randn(2, 3, 64)  # [batch, seq_len, embed_dim]
print(x3.shape)
x3 = rearrange(x3, "batch seq_len (num_heads head_dim) -> batch num_heads seq_len head_dim", head_dim=8)
print(x3.shape)


torch.Size([2, 3, 64])
torch.Size([2, 8, 3, 8])


Drill 4. x has shape [batch, num_heads, seq_len, head_dim] (the output of multi-head attention). Merge the heads back: [batch, seq_len, num_heads * head_dim].

In [37]:
x4 = torch.randn(2, 12, 13, 64) # [batch, num_heads, seq_len, head_dim]
print(x4.shape)
x4 = rearrange(x4, "batch num_heads seq_len head_dim -> batch seq_len (num_heads head_dim)")
x4.shape

torch.Size([2, 12, 13, 64])


torch.Size([2, 13, 768])

Drill 5. x has shape [batch, channels, height, width]. Compute the mean over the spatial dimensions to get [batch, channels] (global average pooling).

In [38]:
x5 = torch.randn(2, 3, 32, 32) # [batch, channels, height, width]
print(x5.shape)
x5 = reduce(x5, "batch channels height width ->  batch channels", "mean")
x5.shape

torch.Size([2, 3, 32, 32])


torch.Size([2, 3])

## Part 2: The Patch Embedding Problem (the one that actually matters for transformers)

This is the operation ViT (Vision Transformer) uses to turn an image into a sequence of patches — and it's a great test of whether you actually understand rearrange's splitting syntax.

Task: x has shape [batch, channels, height, width], where height and width are both divisible by patch_size. Convert it into a sequence of flattened patches: [batch, num_patches, patch_dim], where:

- num_patches = (height / patch_size) * (width / patch_size)
- patch_dim = channels * patch_size * patch_size

In [62]:
x6 = torch.randn(2, 3, 128, 128) # [batch, channels, height, width]
patch_size = 4
x6 = rearrange(x6, "batch channels (height_patches height_size) (width_patches width_size) -> batch (height_patches width_patches) (channels height_size width_size)",height_size=patch_size, width_size=patch_size)

x6.shape

torch.Size([2, 1024, 48])

## Part 3: Debug-the-Pattern

Below are three rearrange calls with subtle bugs. For each, predict what error or wrong-shape output you'd get, then run it and check.

In [67]:
# Bug A: swapped axis order
x = torch.randn(2, 3, 4, 5)  # [batch, channels, height, width]
y = rearrange(x, 'b c h w -> b w h c')  # is this NHWC? check carefully
print(y.shape)

# Bug B: mismatched split
x = torch.randn(2, 12, 8)  # [batch, seq, embed_dim], embed_dim=12, but...
y = rearrange(x, 'b s (h d) -> b h s d', h=2)  # does 12 divide evenly by 5?

# Bug C: forgetting to specify a split size
x = torch.randn(2, 16, 8)  # [batch, seq_len, embed_dim]
y = rearrange(x, 'b s (h d) -> b h s d', h=2)  # what happens with no h= or d= given?

torch.Size([2, 5, 4, 3])
